#### 03 -- Date Dim 

In [1]:
# Imports and setup

from pathlib import Path 
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

project_root = Path.cwd().parent
raw_dir = project_root / "data" / "raw"

In [2]:
# Initialize Spark Session

spark = (
    SparkSession.builder
    .appName("CLVSourceProfiling")
    .master("local[*]")
    .getOrCreate()
)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/09/23 14:21:26 WARN Utils: Your hostname, SamDep, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/09/23 14:21:26 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/23 14:21:27 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/09/23 14:21:29 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [3]:
date_dim_tbl = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(str(raw_dir / "date_dim.csv"))
)

date_dim_tbl.printSchema()

date_dim_tbl.show(10, truncate=False)

print(f"Total records: {date_dim_tbl.count():,}")

root
 |-- date_key: string (nullable = true)
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- week: integer (nullable = true)
 |-- day_of_week: string (nullable = true)
 |-- is_weekend: boolean (nullable = true)
 |-- is_holiday: boolean (nullable = true)
 |-- holiday_name: string (nullable = true)

+----------+----+-----+----+-----------+----------+----------+-----------------------+
|date_key  |year|month|week|day_of_week|is_weekend|is_holiday|holiday_name           |
+----------+----+-----+----+-----------+----------+----------+-----------------------+
|01-01-2023|2023|1    |52  |Sunday     |true      |true      |New Year's Day         |
|02-01-2023|2023|1    |1   |Monday     |false     |true      |New Year's Day Observed|
|03-01-2023|2023|1    |1   |Tuesday    |false     |false     |NULL                   |
|04-01-2023|2023|1    |1   |Wednesday  |false     |false     |NULL                   |
|05-01-2023|2023|1    |1   |Thursday   |false     |false    

In [4]:
date_dim_tbl.select(
    F.count("*").alias("total_rows"),
    F.countDistinct("date_key").alias("distinct_dates"),
    F.min(F.to_date("date_key", "dd-MM-yyyy")).alias("min_date"),
    F.max(F.to_date("date_key", "dd-MM-yyyy")).alias("max_date"),
    F.sum(
        F.when(
            F.to_date("date_key", "dd-MM-yyyy").isNull(),
            1
        ).otherwise(0)
    ).alias("invalid_dates")
).show()

+----------+--------------+----------+----------+-------------+
|total_rows|distinct_dates|  min_date|  max_date|invalid_dates|
+----------+--------------+----------+----------+-------------+
|       365|           365|2023-01-01|2023-12-31|            0|
+----------+--------------+----------+----------+-------------+



In [ ]:
# confirming null values across, expected in holiday_name field - sales trends, etc.
date_dim_tbl.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in date_dim_tbl.columns
]).show()

+--------+----+-----+----+-----------+----------+----------+------------+
|date_key|year|month|week|day_of_week|is_weekend|is_holiday|holiday_name|
+--------+----+-----+----+-----------+----------+----------+------------+
|       0|   0|    0|   0|          0|         0|         0|         353|
+--------+----+-----+----+-----------+----------+----------+------------+



In [9]:
date_check = (
    date_dim_tbl
    .withColumn("parsed_date", F.to_date("date_key", "dd-MM-yyyy"))
    .filter(
        (F.col("year") != F.year("parsed_date"))
        | (F.col("month") != F.month("parsed_date"))
        | (F.col("day_of_week") != F.date_format("parsed_date", "EEEE"))
        | (F.col("is_weekend") != F.dayofweek("parsed_date").isin(1, 7))
        | (F.col("is_holiday") & F.col("holiday_name").isNull())
        | (~F.col("is_holiday") & F.col("holiday_name").isNotNull())
    )
)

print("Inconsistent calendar records:", date_check.count())
date_check.show(10, truncate=False)

Inconsistent calendar records: 0
+--------+----+-----+----+-----------+----------+----------+------------+-----------+
|date_key|year|month|week|day_of_week|is_weekend|is_holiday|holiday_name|parsed_date|
+--------+----+-----+----+-----------+----------+----------+------------+-----------+
+--------+----+-----+----+-----------+----------+----------+------------+-----------+



26/09/24 16:00:22 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 4925625 ms exceeds timeout 120000 ms
26/09/24 16:00:25 WARN SparkContext: Killing executors is not supported by current scheduler.
26/09/24 16:00:28 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:70)
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:44)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:359)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:34)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:132)
	at org.apache.spark.stor